# RG3 gas: learning-strategy ensembles
Only gas17 + normal566 RH. Existing four jitter and four normal-relation residual features unchanged.
Five learning strategies: 100-tree control, positive-block balancing, training hard-negative reweighting, two positive-mode experts with max/mean. Two feature views ensembled by max.
Same outer and inner block splits, internal normal99% calibration. Train-only strategy selector. Many repeated experiments are exploratory, not independent validation.
No commit/push/Confluence changes.

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np,pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import average_precision_score,confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
nb=json.loads((root/'notebooks/22_rg3_nested_signal_ensemble.ipynb').read_text(encoding='utf-8'))
exec(''.join(nb['cells'][1]['source']).split('scores=[];thresholds=[];foldlogs=[]')[0])
variants=['uniform_control','positive_block_balance','hard_negative','gas_mode_max','gas_mode_mean']
pcache={}
def forest():
    return RandomForestClassifier(n_estimators=100,max_depth=8,min_samples_leaf=3,class_weight='balanced_subsample',random_state=42,n_jobs=2)
def predict(ref,q):
    key=(tuple(ref),tuple(q))
    if key in pcache:return pcache[key]
    assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
    if len(np.unique(y[ref]))<2:return {n:np.zeros(len(q)) for n in variants}
    rr=np.full((len(ref),4),np.nan)
    for b in groups.iloc[ref].unique():
        mask=groups.iloc[ref].eq(b).to_numpy()
        if (~mask).any() and (y[ref[~mask]]==0).any():rr[mask]=res_features(ref[~mask],ref[mask])
    views=[(d.iloc[ref][stdcols].to_numpy(float),d.iloc[q][stdcols].to_numpy(float)),(rr,res_features(ref,q))]
    yy=y[ref];gg=groups.iloc[ref].to_numpy();positive=yy==1;normal=yy==0
    weight=np.ones(len(ref))
    for block in np.unique(gg[positive]):
        mask=positive&(gg==block)
        weight[mask]=positive.sum()/(len(np.unique(gg[positive]))*mask.sum())
    outputs={n:[] for n in variants}
    for tr,te in views:
        imp=SimpleImputer(strategy='median',keep_empty_features=True)
        tr=imp.fit_transform(tr);te=imp.transform(te)
        uniform=forest().fit(tr,yy)
        outputs['uniform_control'].append(uniform.predict_proba(te)[:,1])
        balanced=forest().fit(tr,yy,sample_weight=weight)
        outputs['positive_block_balance'].append(balanced.predict_proba(te)[:,1])
        # Hard-normal mining is confined to training data and refit inside each held-out fold.
        # In-sample probabilities are used only to form training weights, NEVER as calibration scores.
        fitted=balanced.predict_proba(tr)[:,1]
        hard=weight.copy()
        boundary=float(np.quantile(fitted[normal],.9))
        hard[normal&(fitted>=boundary)]=3.
        hard[normal]=hard[normal]/hard[normal].mean()
        second=forest().fit(tr,yy,sample_weight=hard)
        outputs['hard_negative'].append(second.predict_proba(te)[:,1])
        # Positive modes are learned only from the current training positives.
        sc=RobustScaler().fit(tr[normal]);z=sc.transform(tr)
        k=min(2,len(np.unique(z[positive],axis=0)))
        if k>1:
            labels=KMeans(n_clusters=k,random_state=42,n_init=10).fit_predict(z[positive])
        else:labels=np.zeros(positive.sum(),int)
        ids=np.flatnonzero(positive);experts=[]
        for mode in np.unique(labels):
            chosen=np.r_[np.flatnonzero(normal),ids[labels==mode]]
            m=forest().fit(tr[chosen],yy[chosen])
            experts.append(m.predict_proba(te)[:,1])
        outputs['gas_mode_max'].append(np.max(experts,axis=0))
        outputs['gas_mode_mean'].append(np.mean(experts,axis=0))
    pcache[key]={n:np.max(v,axis=0) for n,v in outputs.items()}
    return pcache[key]
records=[];choices=[];folds=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
    for j,b in enumerate(blocks):
        q=np.flatnonzero(groups.eq(b))
        ref=np.flatnonzero(~groups.eq(b)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
        if len(np.unique(y[ref]))<2:continue
        outer=predict(ref,q);inner={n:np.full(len(ref),np.nan) for n in variants}
        for ib in groups.iloc[ref].unique():
            mask=groups.iloc[ref].eq(ib).to_numpy()
            if (~mask).any():
                pp=predict(ref[~mask],ref[mask])
                for n in variants:inner[n][mask]=pp[n]
        candidate=[]
        for name in variants:
            cal=inner[name];valid=np.isfinite(cal);normal=cal[(y[ref]==0)&valid]
            cut=float(np.quantile(normal,.99,method='higher')) if len(normal) else 1.
            pr=cal[valid]>cut;yy=y[ref][valid]
            tn,fp,fn,tp=confusion_matrix(yy,pr,labels=[0,1]).ravel()
            f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
            candidate.append(dict(model=name,threshold=cut,inner_F1=f1,inner_FP=int(fp),inner_TP=int(tp)))
            for k,i in enumerate(q):records.append(dict(scheme=scheme,model=name,TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(outer[name][k]),threshold=cut,prediction=int(outer[name][k]>cut)))
        # Prespecified train-only selection among learning strategies; never outer labels.
        selected=max(candidate,key=lambda c:(c['inner_F1'],-c['inner_FP'],c['model']=='uniform_control'))
        for k,i in enumerate(q):records.append(dict(scheme=scheme,model='inner_selected',TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(outer[selected['model']][k]),threshold=selected['threshold'],prediction=int(outer[selected['model']][k]>selected['threshold'])))
        choices.append(dict(scheme=scheme,block=b,**selected))
        folds.append(dict(scheme=scheme,block=b,train=len(ref),train_gas=int(y[ref].sum()),test=len(q),test_gas=int(y[q].sum())))
        print('Finished learning-strategy ensemble',scheme,b,flush=True)
pred=pd.DataFrame(records);rows=[]
for (scheme,name),g in pred.groupby(['scheme','model'],sort=False):
    tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
    rows.append(dict(scheme=scheme,model=name,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.,AP=average_precision_score(g.actual,g.score)))
metrics=pd.DataFrame(rows)
pred.to_csv(tables/'rg3_case_weighting_predictions.csv',index=False,encoding='utf-8-sig')
metrics.to_csv(tables/'rg3_case_weighting_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(choices).to_csv(tables/'rg3_case_weighting_inner_choices.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(folds).to_csv(tables/'rg3_case_weighting_fold_audit.csv',index=False,encoding='utf-8-sig')
print(metrics.to_string(index=False))


## Executed results
Train-only strategy selection: whole TP5 FP14 FN12 TN552 F1=0.278; forward TP3 FP6 FN10 TN421 F1=0.273. Additional gas 2020-10-23 07:21:20. Whole balance target not achieved.
Control uses 100 trees, original baseline 200 retained separately. See iterative improvement log and case_weighting tables.